# Task 2 — Data Collection & Preprocessing

Public source: **UCI Adult (Census Income)**  
https://archive.ics.uci.edu/dataset/2/adult

This notebook collects more than 1,000 real samples, cleans missing values / duplicates / types / outliers, engineers features, and writes the assignment deliverables.

In [ ]:
from pathlib import Path
import sys

import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT / "src"))
from preprocess import (
    CATEGORICAL,
    FIG_DIR,
    PROCESSED_DIR,
    clean_types_and_missing,
    engineer_features,
    handle_outliers,
    load_raw,
    profile_frame,
    run,
)

sns.set_theme(style="whitegrid")
pd.set_option("display.max_columns", 50)
print("Project root:", ROOT)

## 1. Collect the public dataset

UCI publishes `adult.data` (training) and `adult.test` (test). They are concatenated so the working table is well above the 1,000-row minimum.

In [ ]:
raw = load_raw()
print(raw.shape)
raw.head()

In [ ]:
raw.info()
display(raw.describe(include="all").T.head(20))

## 2. Quality issues in the raw file

- Categorical fields are padded with spaces.
- Missing workclass / occupation / native-country are stored as `?`.
- The test file suffixes income with a period (`>50K.`).
- Duplicate rows exist.
- `capital_gain` is top-coded at 99999; `hours_per_week` has long tails.

In [ ]:
preview = raw.copy()
for col in CATEGORICAL + ["income"]:
    preview[col] = preview[col].astype(str).str.strip().replace("?", pd.NA)

print("Duplicate rows:", int(raw.duplicated().sum()))
print("Missing after treating '?' as NA:")
print(preview.isna().sum()[lambda s: s > 0])
print("\nIncome labels:", preview["income"].value_counts(dropna=False).to_dict())

## 3. Clean types, missing values, duplicates, outliers

In [ ]:
cleaned = clean_types_and_missing(raw)
cleaned, outlier_notes = handle_outliers(cleaned)
print(cleaned.shape)
print(outlier_notes)
print("Remaining missing cells:", int(cleaned.isna().sum().sum()))
cleaned.dtypes

## 4. Feature engineering

In [ ]:
featured = engineer_features(cleaned)
new_cols = [c for c in featured.columns if c not in raw.columns]
print("Engineered columns:", new_cols)
featured[new_cols].head()

## 5. Run the full pipeline (writes CSV, figures, reports)

In [ ]:
final = run()
print("Clean rows:", len(final))
print("Clean columns:", list(final.columns))
print("Figures:", sorted(p.name for p in FIG_DIR.glob("*.png")))
print("Processed CSVs:", sorted(p.name for p in PROCESSED_DIR.glob("*.csv")))

## 6. Output figures (assignment screenshots)

In [ ]:
from IPython.display import Image, display

for path in sorted(FIG_DIR.glob("*.png")):
    print(path.name)
    display(Image(filename=str(path)))

## 7. Clean-data check against the brief

- Samples ≥ 1,000
- Missing values handled
- Duplicates removed
- Types corrected
- Outliers treated with documented rules
- Features engineered and saved

In [ ]:
before = profile_frame(preview, "raw")
after = profile_frame(final, "clean")
checks = pd.DataFrame(
    [
        ["n_rows", before["n_rows"], after["n_rows"]],
        ["n_cols", before["n_cols"], after["n_cols"]],
        ["duplicates", before["duplicate_rows"], after["duplicate_rows"]],
        ["missing_cells", before["missing_cells"], after["missing_cells"]],
    ],
    columns=["metric", "before", "after"],
)
display(checks)
assert after["n_rows"] >= 1000
assert after["missing_cells"] == 0
assert after["duplicate_rows"] == 0
print("All assignment checks passed.")